In [ ]:
%pip install -r requirements.txt

In [ ]:
from get_dataset import get_dataset
from ingest import ingest
from config_IO import save_json_config, load_json_config, make_YOLO26_config
from system_info import print_system_info
from view_file import view_image, view_text

from dotenv import load_dotenv
from pathlib import Path
import os
import sys
import platform
import json
import multiprocessing
import yaml
import torch
from ultralytics import YOLO

In [ ]:
USE_KAGGLE = False

load_dotenv()
KAGGLE_USERNAME = os.getenv('KAGGLE_USERNAME')
KAGGLE_KEY = os.getenv('KAGGLE_KEY')

get_dataset(from_kaggle=USE_KAGGLE, kaggle_username=KAGGLE_USERNAME, kaggle_key=KAGGLE_KEY)

In [ ]:
ingest()

In [ ]:
make_YOLO26_config()

In [ ]:
print_system_info()

In [ ]:
## CPU Usage Config


CPU_THREADS = 4

os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)

torch.set_num_threads(CPU_THREADS)

print("Using CPU threads:", CPU_THREADS)

In [ ]:
## Checks ALL system configs for current settings.
## Also catches many common setup errors or issues.


# ==========================
# USER CONFIG
# ==========================
DATA_YAML = Path("/workspace/flower-classifier/config/data.yaml")
DATASET_ROOT = Path("/workspace/data/data_set/tomato-flower")
PROJECT_DIR = Path("/workspace/flower-classifier/runs")
RUN_NAME = "tomato_flower_yolo26_from_scratch"
MODEL_CFG = "yolo26n.yaml"   # change later if needed

# ==========================
# HELPERS
# ==========================
def count_files(path, exts=None):
    if not path.exists():
        return 0
    files = [p for p in path.iterdir() if p.is_file()]
    if exts:
        exts = {e.lower() for e in exts}
        files = [p for p in files if p.suffix.lower() in exts]
    return len(files)

def print_path_status(label, path):
    path = Path(path)
    print(f"{label}:")
    print(f"  path   = {path}")
    print(f"  exists = {path.exists()}")
    print(f"  is_dir = {path.is_dir()}")
    print(f"  is_file= {path.is_file()}")
    print()

# ==========================
# SYSTEM INFO
# ==========================
print("=== SYSTEM INFO ===")
print("Python executable:", sys.executable)
print("Python version   :", sys.version.splitlines()[0])
print("Platform         :", platform.platform())
print("Machine          :", platform.machine())
print("Processor        :", platform.processor())
print("CPU count        :", multiprocessing.cpu_count())
print("Working dir      :", Path.cwd())
print()

print("=== TORCH INFO ===")
print("torch version    :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())
print("MPS available    :", hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
print("torch num threads:", torch.get_num_threads())
print()

# ==========================
# ENV VARS
# ==========================
print("=== IMPORTANT ENV VARS ===")
for key in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "CUDA_VISIBLE_DEVICES",
    "ULTRALYTICS_CONFIG_DIR",
]:
    print(f"{key} =", os.environ.get(key))
print()

# ==========================
# PATH CHECKS
# ==========================
print("=== PATH CHECKS ===")
print_path_status("DATA_YAML", DATA_YAML)
print_path_status("DATASET_ROOT", DATASET_ROOT)
print_path_status("PROJECT_DIR", PROJECT_DIR)

for split in ["train", "val", "test"]:
    split_dir = DATASET_ROOT / split
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"

    print_path_status(f"{split.upper()} DIR", split_dir)
    print(f"{split.upper()} COUNTS:")
    print("  image files =", count_files(img_dir, {".jpg", ".jpeg", ".png", ".bmp", ".webp"}))
    print("  label files =", count_files(lbl_dir, {".txt"}))
    print()

# ==========================
# YAML CHECK
# ==========================
print("=== YAML CONTENTS ===")
with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

print(json.dumps(data_cfg, indent=2))

print()
print("=== YAML PATH VALIDATION ===")
for key in ["train", "val", "test"]:
    p = Path(data_cfg[key])
    print(f"{key}: {p} | exists={p.exists()} | is_dir={p.is_dir()}")

In [ ]:
## Verify YOLO26 is accessible and usable

try:
    YOLO("yolo26n.yaml")
    print("DONE! Model config recognized")
except Exception as e:
    print("ERROR! Model config NOT recognized")
    print(e)

In [ ]:
## Load parameters from specified file location

file_location = "/workspace/flower-classifier/config/YOLO26_config.json"

cfg = load_json_config(file_location)

MODEL_CFG = cfg["model_cfg"]
DATA_YAML = cfg["data_yaml"]
PROJECT_DIR = cfg["project_dir"]
RUN_NAME = cfg["run_name"]
TRAIN_PARAMS = cfg["train_params"]

print("MODEL_CFG =", MODEL_CFG)
print("DATA_YAML =", DATA_YAML)
print("PROJECT_DIR =", PROJECT_DIR)
print("RUN_NAME =", RUN_NAME)
print("TRAIN_PARAMS =", TRAIN_PARAMS)

In [ ]:
## Manual overrides

## for example:
# cfg["run_name"] = "tomato_flower_yolo26_trial_02"
# cfg["train_params"]["epochs"] = 150

## We change the configs within the 'cfg' object as opposed to directly on MODEL_CFG or other imported vars.
## This is so, whenever we save the configs, they can be exported back to the JSON config file if we wish.

In [ ]:
## Training the YOLO26 model

model = YOLO(cfg["model_cfg"])

results = model.train(
    data=cfg["data_yaml"],
    project=cfg["project_dir"],
    name=cfg["run_name"],
    **cfg["train_params"]
)

print("DONE! Training complete!")

In [ ]:
## Save current config values backk to specified config json file.

CONFIG_PATH = "/workspace/flower-classifier/config/YOLO26_config.json"
save_json_config(cfg, CONFIG_PATH)